# Stiffness 5 — lumping the particle, or resolving it

The same column and the same linear rapid-equilibrium isotherm as the previous notebook. The question here is the one every CADET user eventually asks:

> **Can I use the lumped rate model instead of the general rate model to go faster?**

LRMP treats a particle as one well mixed volume behind a film resistance. The GRM resolves the radial direction. LRMP is exactly the limit of the GRM for infinitely fast pore diffusion — so the two describe *the same column* whenever the particle really is uniform, and **different columns** whenever it is not.

Whether it is uniform is decided by how far the solute reaches into the particle while the bulk is changing, $\delta \approx \sqrt{D_p t}$:

| $D_p$ | $\delta$ / radius | the particle is |
|---|---|---|
| 10⁻⁹ | 9.9 | uniform — lumping is legitimate |
| 10⁻¹¹ | 1.0 | partly penetrated — the realistic case |
| 10⁻¹³ | 0.1 | a thin shell over a dead core |

So the knobs are **what you resolve** and **whether there is anything to resolve**.

In [ ]:
import utility.particle_model_explorer as explorer

explorer.show_setting()

## What to watch

Every row is scored against its **own** converged general rate model at the same pore diffusivity, so the error column is the price of lumping — not the gap between two unrelated models.

Three columns matter. The **error** says whether the lumped model is describing your column at all. The **step count** says whether the time integration got easier. The **compute time** says what you actually saved. They do not agree, and the disagreement is the point.

In [ ]:
from ipywidgets import interact

show_particle = explorer.make_explorer()

interact(show_particle, **show_particle.sliders);

## What happened

**Lumping does not make the time integration easier. It makes it harder.**

At the realistic diffusivity, $D_p = 10^{-11}$:

| model | unknowns per cell | BDF steps | error | seconds |
|---|---|---|---|---|
| LRMP | 3 | **2023** | **7.74 %** | 0.23 |
| GRM, 1 cell | 3 | 2920 | 2.89 % | 0.35 |
| GRM, 16 cells | 33 | **1465** | 0.03 % | 1.44 |
| GRM, 128 cells | 257 | 1427 | — | 18.7 |

The lumped model takes **40 % more steps** than a well resolved general rate model, and is 7.7 % wrong. What it does save is degrees of freedom, and that is where its six-fold advantage in seconds comes from — not from an easier problem.

**And the accuracy of lumping is decided entirely by the penetration depth.** The same LRMP simulation — it has no $D_p$, so it is literally the same run — is scored against each reference:

| $\delta$ / radius | LRMP error |
|---|---|
| 9.9 (uniform) | 0.43 % |
| 1.0 (realistic) | 7.74 % |
| 0.1 (shell) | **16.5 %** |

Lumping is not a numerical approximation you can refine away. It is a modelling assumption, and it is either true of your resin or it is not.

<details>
<summary><b>Click to learn more: an oddity we measured and cannot explain</b></summary>

Push the pore diffusivity up to $10^{-7}$ and the general rate model becomes the lumped model: the two chromatograms agree to **0.005 %**, with identical peak height and identical width. The physics is the same.

They still do not cost the same. LRMP takes 2023 steps; a GRM with a single radial cell — the same three unknowns per axial position, the same solution — takes **1465**. Adding radial cells barely changes it: 64 cells, with 129 unknowns per axial position, takes 1416.

So the gap is not the number of variables, and it is not the solution. Two formulations that should be mathematically equivalent integrate differently by nearly 40 %, and we have not been able to account for it from the outside. It is recorded here as a measurement rather than explained, and it is worth raising with the CADET developers.

</details>

> **Take-away.** Use LRMP when your particle is genuinely uniform — check $\sqrt{D_p t}$ against the radius before you decide, not after. You will save degrees of freedom and wall clock, you will not save time steps, and if the particle is not uniform you are not approximating the answer, you are changing the question. In this column, at a realistic pore diffusivity, lumping was 7.7 % wrong for a sixfold speed-up, and resolving the particle with 16 radial cells cost 1.4 s.